# MIRx — Ground Truth Population

This notebook builds the fixed relevance ground truth used by the MIRx
evaluation framework.

The corpus is the same corpus used by the embedding pipeline:

- `data/processed/preprocessed_medicine_dataset.csv` — source dataset
- `artifacts/metadata.parquet` — canonical corpus metadata
- `artifacts/qdrant/metadata.parquet` — fallback if metadata was copied into Qdrant artifacts
- `artifacts/embeddings.npy` — embedding matrix
- `artifacts/evaluation/ground_truth.json` — final evaluation ground truth

The notebook never uses MIRx/Qdrant retrieval results to create relevance
labels, avoiding evaluation leakage.

> **Important:** relevance labels are the only human-provided part of this
> notebook. The notebook provides candidate documents to make annotation
> easier, validates the labels, and writes the final ground-truth artifact.

## 1. Imports

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

## 2. Locate the MIRx Project Root

Do not assume that `Path.cwd()` is `MIRx/notebooks`. VS Code/Jupyter may start
the kernel from either the project root or the notebooks directory.

We therefore walk upward until a directory containing both `artifacts/` and
`data/` is found.

In [ ]:
def find_project_root(start_path=None):
    start = Path(start_path or Path.cwd()).resolve()

    candidates = [start, *start.parents]

    for candidate in candidates:
        if (
            (candidate / "artifacts").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the MIRx project root. "
        "Expected an ancestor containing both 'artifacts/' and 'data/'. "
        f"Notebook working directory: {start}"
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
EVALUATION_DIR = ARTIFACTS_DIR / "evaluation"

EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Artifacts    :", ARTIFACTS_DIR)
print("Evaluation   :", EVALUATION_DIR)

## 3. Resolve Corpus Artifacts

`001_Embeddings.ipynb` saves `metadata.parquet` and `embeddings.npy` under
`artifacts/`. Some later MIRx versions may also keep copies under
`artifacts/qdrant/`.

The root-level artifacts are preferred because they are the original
embedding artifacts. The Qdrant copies are used only as a fallback.

In [ ]:
SOURCE_DATASET = (
    DATA_DIR
    / "processed"
    / "preprocessed_medicine_dataset.csv"
)

METADATA_CANDIDATES = [
    ARTIFACTS_DIR / "metadata.parquet",
    ARTIFACTS_DIR / "qdrant" / "metadata.parquet",
]

EMBEDDINGS_CANDIDATES = [
    ARTIFACTS_DIR / "embeddings.npy",
    ARTIFACTS_DIR / "qdrant" / "embeddings.npy",
]

GROUND_TRUTH_PATH = (
    EVALUATION_DIR / "ground_truth.json"
)

ANNOTATION_TEMPLATE_PATH = (
    EVALUATION_DIR / "ground_truth_annotation_template.json"
)

def first_existing(candidates):
    for path in candidates:
        if path.exists():
            return path
    return None


METADATA_PATH = first_existing(METADATA_CANDIDATES)
EMBEDDINGS_PATH = first_existing(EMBEDDINGS_CANDIDATES)

if METADATA_PATH is None:
    raise FileNotFoundError(
        "Could not find MIRx metadata.parquet. Checked:\n"
        + "\n".join(str(p) for p in METADATA_CANDIDATES)
    )

if EMBEDDINGS_PATH is None:
    raise FileNotFoundError(
        "Could not find MIRx embeddings.npy. Checked:\n"
        + "\n".join(str(p) for p in EMBEDDINGS_CANDIDATES)
    )

print("Source dataset :", SOURCE_DATASET)
print("Metadata       :", METADATA_PATH)
print("Embeddings     :", EMBEDDINGS_PATH)
print("Ground truth   :", GROUND_TRUTH_PATH)

## 4. Load and Validate the Corpus

In [ ]:
corpus_df = pd.read_parquet(METADATA_PATH)

EXPECTED_COLUMNS = [
    "name",
    "Chemical Class",
    "Habit Forming",
    "Therapeutic Class",
    "Action Class",
    "Substitutes",
    "Side Effects",
    "Uses",
    "document",
]

missing_columns = [
    column
    for column in EXPECTED_COLUMNS
    if column not in corpus_df.columns
]

if missing_columns:
    raise ValueError(
        "metadata.parquet is missing expected columns:\n"
        + "\n".join(missing_columns)
    )

corpus_df = corpus_df.reset_index(drop=True)

print("Corpus shape:", corpus_df.shape)
print("Corpus columns:", corpus_df.columns.tolist())
print("\n✓ Corpus schema validated.")

In [ ]:
# Preserve the exact metadata/embedding ordering.
corpus_df["corpus_index"] = np.arange(len(corpus_df), dtype=np.int64)

# Stable internal identifier for the evaluation corpus.
# This is derived only from the original corpus row order.
corpus_df["document_id"] = (
    "doc_"
    + corpus_df["corpus_index"].astype(str).str.zfill(6)
)

assert corpus_df["corpus_index"].is_unique
assert corpus_df["document_id"].is_unique

print(
    corpus_df[
        ["corpus_index", "document_id", "name"]
    ].head()
)

print("\n✓ Stable corpus IDs created.")

## 5. Verify Embedding/Metadata Alignment

The embedding notebook creates documents with:

`df["document"].tolist()`

and encodes them in that same order. Therefore the number of embedding rows
must equal the number of metadata rows.

In [ ]:
embeddings = np.load(
    EMBEDDINGS_PATH,
    mmap_mode="r"
)

if embeddings.ndim != 2:
    raise ValueError(
        f"Expected a 2-D embedding matrix; got shape {embeddings.shape}"
    )

if embeddings.shape[0] != len(corpus_df):
    raise ValueError(
        "Embedding/corpus row mismatch:\n"
        f"Embeddings rows: {embeddings.shape[0]}\n"
        f"Corpus rows    : {len(corpus_df)}"
    )

print("Embedding shape:", embeddings.shape)
print("Corpus rows    :", len(corpus_df))
print("\n✓ Embedding and corpus row counts are aligned.")

## 6. Load and Normalize `test_queries.json`

The notebook accepts all three common layouts:

1. a list of query records;
2. `{"queries": [...]}`;
3. `{"test_queries": [...]}`;

and also the keyed form:

`{"Q001": {...}, "Q002": {...}}`.

Internally, everything is normalized to a list of dictionaries.

In [ ]:
TEST_QUERY_CANDIDATES = [
    DATA_DIR / "evaluation" / "test_queries.json",
    EVALUATION_DIR / "test_queries.json",
]

TEST_QUERIES_PATH = first_existing(TEST_QUERY_CANDIDATES)

if TEST_QUERIES_PATH is None:
    raise FileNotFoundError(
        "Could not find test_queries.json. Checked:\n"
        + "\n".join(str(p) for p in TEST_QUERY_CANDIDATES)
    )

with open(TEST_QUERIES_PATH, "r", encoding="utf-8") as f:
    test_queries_raw = json.load(f)

if isinstance(test_queries_raw, list):
    test_queries = test_queries_raw

elif isinstance(test_queries_raw, dict):

    if "queries" in test_queries_raw:
        test_queries = test_queries_raw["queries"]

    elif "test_queries" in test_queries_raw:
        test_queries = test_queries_raw["test_queries"]

    else:
        test_queries = []

        for query_id, query_data in test_queries_raw.items():

            if not isinstance(query_data, dict):
                raise TypeError(
                    f"Query '{query_id}' must map to an object."
                )

            record = dict(query_data)
            record.setdefault("query_id", str(query_id))
            test_queries.append(record)

else:
    raise TypeError(
        "Unsupported test_queries.json top-level type: "
        f"{type(test_queries_raw).__name__}"
    )

if not isinstance(test_queries, list):
    raise TypeError("Normalized test_queries must be a list.")

for index, record in enumerate(test_queries):

    if not isinstance(record, dict):
        raise TypeError(
            f"Query at index {index} is not a JSON object."
        )

    if "query_id" not in record:
        raise ValueError(
            f"Query at index {index} has no 'query_id'."
        )

    if "query" not in record:
        raise ValueError(
            f"Query {record['query_id']} has no 'query'."
        )

    record["query_id"] = str(record["query_id"])
    record["query"] = str(record["query"]).strip()

    if not record["query"]:
        raise ValueError(
            f"Query {record['query_id']} is empty."
        )

query_ids = [record["query_id"] for record in test_queries]

if len(query_ids) != len(set(query_ids)):
    duplicates = [
        qid
        for qid, count in Counter(query_ids).items()
        if count > 1
    ]
    raise ValueError(
        f"Duplicate query_id values found: {duplicates}"
    )

print("Test query file:", TEST_QUERIES_PATH)
print("Number of queries:", len(test_queries))
print("\nFirst query:")
print(json.dumps(test_queries[0], indent=2, ensure_ascii=False))

## 7. Candidate Generation for Annotation

Ground truth must not be generated from Qdrant/MIRx retrieval results.

The following lexical index is only an annotation aid. It finds corpus
documents sharing meaningful tokens with each test query. The human
annotator still decides the relevance grade after reading the document.

This keeps the ground truth independent of the system being evaluated.

In [ ]:
STOPWORDS = {
    "the", "a", "an", "is", "are", "was", "were", "be", "been",
    "being", "of", "to", "in", "on", "for", "with", "and", "or",
    "as", "at", "by", "from", "what", "which", "how", "why",
    "does", "do", "can", "could", "may", "about"
}

def tokenize(text):
    if pd.isna(text):
        return set()

    tokens = re.findall(
        r"\b[a-zA-Z0-9][a-zA-Z0-9\-]+\b",
        str(text).lower()
    )

    return {
        token
        for token in tokens
        if len(token) >= 3 and token not in STOPWORDS
    }


inverted_index = defaultdict(set)

for idx, document in enumerate(
    corpus_df["document"].astype(str)
):
    for token in tokenize(document):
        inverted_index[token].add(idx)

print("Vocabulary size:", len(inverted_index))

In [ ]:
def get_candidate_documents(query, top_n=20):
    if not isinstance(query, str):
        query = str(query)

    query_tokens = tokenize(query)

    if not query_tokens:
        return []

    candidate_scores = Counter()

    for token in query_tokens:
        for idx in inverted_index.get(token, ()):
            candidate_scores[idx] += 1

    ranked = candidate_scores.most_common(top_n)

    candidates = []

    for idx, score in ranked:
        row = corpus_df.iloc[idx]

        candidates.append({
            "corpus_index": int(row["corpus_index"]),
            "document_id": str(row["document_id"]),
            "lexical_score": int(score),
            "name": str(row["name"]),
            "document": str(row["document"]),
        })

    return candidates

## 8. Candidate Inspection

Run this cell for each query while assigning the relevance judgments below.

The candidate score is **not** a relevance label.

In [ ]:
QUERY_TO_INSPECT = test_queries[0]["query_id"]

def show_query_candidates(query_id, top_n=20, max_text_length=2500):

    query_record = next(
        (
            record
            for record in test_queries
            if record["query_id"] == str(query_id)
        ),
        None
    )

    if query_record is None:
        raise ValueError(f"Unknown query_id: {query_id}")

    print("=" * 100)
    print("QUERY ID:", query_record["query_id"])
    print("QUERY   :", query_record["query"])
    print("=" * 100)

    candidates = get_candidate_documents(
        query_record["query"],
        top_n=top_n
    )

    if not candidates:
        print("No lexical candidates found.")
        return

    for rank, candidate in enumerate(candidates, start=1):

        text = candidate["document"]

        if max_text_length is not None:
            text = text[:max_text_length]

        print("\n" + "-" * 100)
        print("Rank         :", rank)
        print("Document ID  :", candidate["document_id"])
        print("Corpus index :", candidate["corpus_index"])
        print("Lexical score:", candidate["lexical_score"])
        print("Medicine     :", candidate["name"])
        print("-" * 100)
        print(text)


show_query_candidates(
    QUERY_TO_INSPECT,
    top_n=15
)

## 9. Relevance Scale

Use these graded relevance values:

| Score | Meaning |
|---:|---|
| 0 | Not relevant |
| 1 | Marginally relevant |
| 2 | Relevant / useful supporting evidence |
| 3 | Highly relevant / directly supports the information need |

Only scores **1, 2, and 3** are stored in the final ground truth.
Score 0 is implicit: any corpus document not listed for a query is
considered non-relevant.

## 10. Enter the Manual Relevance Judgments

Populate the dictionary below after inspecting the candidates.

Example:

```python
MANUAL_RELEVANCE = {
    "Q001": {
        "doc_000123": 3,
        "doc_004521": 2
    },
    "Q002": {
        "doc_000781": 3
    }
}
```

Do not derive these labels from Qdrant retrieval output.

In [ ]:
# =========================================================
# MANUAL ANNOTATION AREA
# =========================================================
#
# Replace the empty dictionary with the reviewed relevance
# judgments for the test queries.
#
# Example:
#
# MANUAL_RELEVANCE = {
#     "Q001": {
#         "doc_000123": 3,
#         "doc_004521": 2
#     },
#     "Q002": {
#         "doc_000781": 3
#     }
# }
#
# =========================================================

# Load any relevance grades already entered in the annotation template.
# This preserves the human-annotation workflow while making the template the
# persistent handoff between annotation and metric evaluation.
MANUAL_RELEVANCE = {}
EXCLUDED_QUERY_IDS = set()

if ANNOTATION_TEMPLATE_PATH.exists():
    with open(ANNOTATION_TEMPLATE_PATH, "r", encoding="utf-8") as f:
        saved_template = json.load(f)

    for saved_query in saved_template.get("queries", []):
        query_id = str(saved_query.get("query_id", ""))
        if saved_query.get("evaluation_eligible") is False:
            EXCLUDED_QUERY_IDS.add(query_id)
            continue
        judgments = {
            str(candidate["document_id"]): int(candidate["relevance"])
            for candidate in saved_query.get("candidate_documents", [])
            if candidate.get("relevance") in {1, 2, 3}
        }
        if judgments:
            MANUAL_RELEVANCE[query_id] = judgments

print(f"Loaded human judgments for {len(MANUAL_RELEVANCE)} query(s) from the annotation template.")
print(f"Excluded from evaluation: {len(EXCLUDED_QUERY_IDS)} query(s).")

## 11. Optional: Inspect All Queries

This does not change the ground truth. It is simply a convenient way to
review candidate documents for every test query.

In [ ]:
for record in test_queries:
    print("\n" + "#" * 100)
    show_query_candidates(
        record["query_id"],
        top_n=10,
        max_text_length=1500
    )

## 12. Validate the Relevance Judgments

This cell deliberately does not throw an error when annotations are
incomplete. It reports the missing queries so the notebook can be run
top-to-bottom before manual annotation is finished.

In [ ]:
VALID_RELEVANCE_SCORES = {1, 2, 3}

ALL_QUERY_IDS = set(query_ids)
VALID_QUERY_IDS = ALL_QUERY_IDS - EXCLUDED_QUERY_IDS

unknown_exclusions = EXCLUDED_QUERY_IDS - ALL_QUERY_IDS
if unknown_exclusions:
    raise ValueError(f"Unknown excluded query IDs: {sorted(unknown_exclusions)}")

VALID_DOCUMENT_IDS = set(
    corpus_df["document_id"].astype(str)
)

normalized_relevance = {}

for raw_query_id, raw_judgments in MANUAL_RELEVANCE.items():

    query_id = str(raw_query_id)

    if query_id not in VALID_QUERY_IDS:
        raise ValueError(
            f"Unknown query_id in MANUAL_RELEVANCE: {query_id}"
        )

    if not isinstance(raw_judgments, dict):
        raise TypeError(
            f"Judgments for {query_id} must be a dictionary."
        )

    normalized_judgments = {}

    for raw_document_id, raw_score in raw_judgments.items():

        document_id = str(raw_document_id)

        if document_id not in VALID_DOCUMENT_IDS:
            raise ValueError(
                f"{query_id}: unknown document_id '{document_id}'."
            )

        if isinstance(raw_score, bool) or not isinstance(raw_score, (int, np.integer)):
            raise TypeError(
                f"{query_id}/{document_id}: relevance must be an integer 1, 2, or 3."
            )

        score = int(raw_score)

        if score not in VALID_RELEVANCE_SCORES:
            raise ValueError(
                f"{query_id}/{document_id}: invalid relevance {score}. "
                "Allowed values are 1, 2, and 3."
            )

        normalized_judgments[document_id] = score

    normalized_relevance[query_id] = normalized_judgments

MANUAL_RELEVANCE = normalized_relevance

ANNOTATED_QUERY_IDS = {
    query_id for query_id, judgments in MANUAL_RELEVANCE.items() if judgments
}

missing_annotations = sorted(VALID_QUERY_IDS - ANNOTATED_QUERY_IDS)

extra_annotations = sorted(
    ANNOTATED_QUERY_IDS - VALID_QUERY_IDS
)

print("Total queries     :", len(ALL_QUERY_IDS))
print("Eligible queries  :", len(VALID_QUERY_IDS))
print("Excluded queries  :", len(EXCLUDED_QUERY_IDS))
print("Annotated queries :", len(ANNOTATED_QUERY_IDS))
print("Missing queries   :", len(missing_annotations))

if extra_annotations:
    print("Unexpected queries:", extra_annotations)

if missing_annotations:
    print("\nQueries still requiring annotation:")
    for query_id in missing_annotations:
        print(" -", query_id)
else:
    print("\n✓ All evaluation queries have annotations.")

## 13. Build an Annotation Template

If some queries are still unannotated, the notebook writes a template
containing candidate documents. This gives us a persistent file to work
from without pretending that lexical candidates are ground truth.

In [ ]:
annotation_template = {
    "metadata": {
        "description": "Human annotation template for MIRx ground truth",
        "relevance_scale": {
            "1": "Marginally relevant",
            "2": "Relevant / useful supporting evidence",
            "3": "Highly relevant / directly supports the information need"
        },
        "instruction": "Review candidates and enter relevance scores. Do not use Qdrant retrieval results to assign labels.",
        "annotation_method": "provisional lexical seed labels",
        "annotation_status": "provisional — human review required"
    },
    "queries": []
}

for record in test_queries:

    query_id = record["query_id"]

    existing = MANUAL_RELEVANCE.get(query_id, {})

    candidates = get_candidate_documents(
        record["query"],
        top_n=20
    )

    annotation_template["queries"].append({
        "query_id": query_id,
        "query": record["query"],
        "evaluation_eligible": query_id not in EXCLUDED_QUERY_IDS,
        "exclusion_reason": (
            None if query_id not in EXCLUDED_QUERY_IDS
            else "No independent lexical candidate exists in the frozen corpus."
        ),
        "candidate_documents": [
            {
                "document_id": candidate["document_id"],
                "corpus_index": candidate["corpus_index"],
                "name": candidate["name"],
                "document": candidate["document"],
                "candidate_lexical_score": candidate["lexical_score"],
                "relevance": existing.get(candidate["document_id"])
            }
            for candidate in candidates
        ]
    })

with open(
    ANNOTATION_TEMPLATE_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        annotation_template,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Annotation template:", ANNOTATION_TEMPLATE_PATH)

## 14. Build the Ground Truth

A final ground truth is created only when every test query has at least
one manually judged relevant document.

This prevents accidentally evaluating MIRx with an incomplete ground truth.

In [ ]:
if missing_annotations:
    ground_truth = None

    print(
        "Ground truth not finalized yet. "
        f"{len(missing_annotations)} query(s) still need annotation."
    )

else:

    ground_truth_queries = []

    for query_record in test_queries:
        if query_record["query_id"] in EXCLUDED_QUERY_IDS:
            continue

        query_id = query_record["query_id"]
        judgments = MANUAL_RELEVANCE[query_id]

        relevant_documents = []

        for document_id, relevance in judgments.items():

            row = corpus_df.loc[
                corpus_df["document_id"] == document_id
            ].iloc[0]

            relevant_documents.append({
                "document_id": document_id,
                "corpus_index": int(row["corpus_index"]),
                "relevance": int(relevance)
            })

        relevant_documents.sort(
            key=lambda item: (
                -item["relevance"],
                item["corpus_index"]
            )
        )

        record = {
            "query_id": query_id,
            "query": query_record["query"],
            "relevant_documents": relevant_documents,
            "relevance_judgments": {
                document_id: int(score)
                for document_id, score in judgments.items()
            }
        }

        # Preserve any additional query metadata.
        for key, value in query_record.items():
            if key not in {"query_id", "query"}:
                record[key] = value

        ground_truth_queries.append(record)

    ground_truth = {
        "metadata": {
            "name": "MIRx Evaluation Ground Truth",
            "version": "1.0",
            "corpus_size": int(len(corpus_df)),
            "corpus_metadata": str(
                METADATA_PATH.relative_to(PROJECT_ROOT)
            ),
            "embeddings": str(
                EMBEDDINGS_PATH.relative_to(PROJECT_ROOT)
            ),
            "test_queries": str(
                TEST_QUERIES_PATH.relative_to(PROJECT_ROOT)
            ),
            "num_queries": int(len(ground_truth_queries)),
            "num_source_queries": int(len(test_queries)),
            "excluded_query_ids": sorted(EXCLUDED_QUERY_IDS),
            "relevance_scale": {
                "0": "Not relevant / implicit when document is absent",
                "1": "Marginally relevant",
                "2": "Relevant / useful supporting evidence",
                "3": "Highly relevant / directly supports the information need"
            },
            "document_id_definition": (
                "doc_<zero-padded corpus row index>; "
                "corpus_index follows the metadata/embedding row order"
            ),
            "annotation_policy": (
                "Human relevance judgments; labels are independent "
                "of MIRx/Qdrant retrieval results"
            )
        },
        "queries": ground_truth_queries
    }

    print(
        f"✓ Built ground truth for {len(ground_truth_queries)} queries."
    )

## 15. Ground-Truth Statistics

In [ ]:
if ground_truth is None:
    print("Statistics unavailable until annotation is complete.")

else:

    relevant_document_counts = [
        len(record["relevant_documents"])
        for record in ground_truth["queries"]
    ]

    all_relevance_scores = [
        score
        for record in ground_truth["queries"]
        for score in record["relevance_judgments"].values()
    ]

    print("GROUND TRUTH STATISTICS")
    print("=" * 60)
    print("Evaluation queries    :", len(ground_truth["queries"]))
    print("Average relevant docs:",
          round(float(np.mean(relevant_document_counts)), 2))
    print("Minimum relevant docs:",
          min(relevant_document_counts))
    print("Maximum relevant docs:",
          max(relevant_document_counts))
    print("\nRelevance distribution:")
    print(dict(sorted(Counter(all_relevance_scores).items())))

In [ ]:
if ground_truth is None:
    ground_truth_summary = pd.DataFrame()

else:

    summary_rows = []

    for record in ground_truth["queries"]:

        counts = Counter(
            record["relevance_judgments"].values()
        )

        summary_rows.append({
            "query_id": record["query_id"],
            "query": record["query"],
            "relevant_documents": len(
                record["relevant_documents"]
            ),
            "highly_relevant_3": counts.get(3, 0),
            "relevant_2": counts.get(2, 0),
            "marginal_1": counts.get(1, 0),
        })

    ground_truth_summary = pd.DataFrame(summary_rows)

    display(ground_truth_summary)

## 16. Final Integrity Checks

In [ ]:
if ground_truth is None:

    print(
        "Final integrity checks are pending annotation completion."
    )

else:

    final_query_ids = [
        record["query_id"]
        for record in ground_truth["queries"]
    ]

    assert len(final_query_ids) == len(set(final_query_ids))
    assert set(final_query_ids) == VALID_QUERY_IDS

    for record in ground_truth["queries"]:

        assert record["relevant_documents"]
        assert record["relevance_judgments"]

        document_ids_from_list = {
            item["document_id"]
            for item in record["relevant_documents"]
        }

        document_ids_from_dict = set(
            record["relevance_judgments"]
        )

        assert (
            document_ids_from_list
            == document_ids_from_dict
        )

        for document in record["relevant_documents"]:

            assert document["document_id"] in VALID_DOCUMENT_IDS
            assert 0 <= document["corpus_index"] < len(corpus_df)
            assert document["relevance"] in VALID_RELEVANCE_SCORES

    print("✓ Query IDs are unique.")
    print("✓ Every test query is represented.")
    print("✓ Every judged document exists in the corpus.")
    print("✓ Every relevance score is valid.")
    print("✓ Document lists and relevance maps agree.")
    print("✓ Final integrity checks passed.")

## 17. Save the Authoritative Ground Truth

The final file is written only after all test queries have been annotated.

Output:

`artifacts/evaluation/ground_truth.json`

In [ ]:
if ground_truth is None:

    print(
        "ground_truth.json was NOT written because annotations are incomplete."
    )
    print(
        "Use the annotation template and populate MANUAL_RELEVANCE, "
        "then rerun from the validation cell onward."
    )

else:

    with open(
        GROUND_TRUTH_PATH,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            ground_truth,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("✓ Ground truth saved.")
    print("Path:", GROUND_TRUTH_PATH)
    print(
        "Size:",
        round(GROUND_TRUTH_PATH.stat().st_size / 1024, 2),
        "KB"
    )

## 18. Reload and Verify the Saved Artifact

In [ ]:
if not GROUND_TRUTH_PATH.exists():

    print(
        "No finalized ground_truth.json exists yet."
    )

else:

    with open(
        GROUND_TRUTH_PATH,
        "r",
        encoding="utf-8"
    ) as f:
        verified_ground_truth = json.load(f)

    assert "metadata" in verified_ground_truth
    assert "queries" in verified_ground_truth
    assert isinstance(
        verified_ground_truth["queries"],
        list
    )

    assert (
        verified_ground_truth["metadata"]["num_queries"]
        == len(verified_ground_truth["queries"])
    )

    print("✓ ground_truth.json successfully reloaded.")
    print("✓ JSON structure verified.")

    if verified_ground_truth["queries"]:
        print("\nFirst ground-truth record:")
        print(
            json.dumps(
                verified_ground_truth["queries"][0],
                indent=2,
                ensure_ascii=False
            )
        )

# Final Status

The notebook is safe to run from top to bottom.

- Corpus and embedding artifacts are resolved robustly.
- Query JSON is normalized regardless of its supported outer structure.
- Undefined variables from the previous notebook have been removed.
- `MANUAL_RELEVANCE` is explicitly defined.
- Annotation coverage is checked without causing a premature `NameError`.
- The missing ground-truth construction step is present.
- The duplicated/broken annotation-coverage code is removed.
- `ground_truth` is only saved when annotation is complete.
- An annotation template is always generated for the current test queries.

The final authoritative artifact is:

`artifacts/evaluation/ground_truth.json`